In [1]:
# ==============================
# EVALUATE ANOMALY DETECTION WITH DIFFERENT THRESHOLDS (AND FOR DIFFERENT MODELS)
# ==============================

!pip install pyod

# upload local file
from google.colab import files

eval_upl = files.upload()  # eval-set.csv in "evaluation" folder
normal_upl = files.upload()  # from here on all in "training" folder
inf_c0_upl = files.upload()
inf_c1_upl = files.upload()
inf_c2_upl = files.upload()
inf_c3_upl = files.upload()
inf_c4_upl = files.upload()
inf_c5_upl = files.upload()

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 KB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pyod: filename=pyod-1.0.7-py3-none-any.whl size=181101 sha256=747b15e7362cf40526965610e40b691652add91f30f7d2f835c1ef926cf5ed24
  Stored in directory: /root/.cache/pip/wheels/f7/e2/c1/1c7fd8b261e72411f6509afb429c84532e40ddcd96074473f4
Successfully built pyod


Saving eval-set.csv to eval-set.csv


Saving normal-behavior.csv to normal-behavior.csv


Saving infected-c0-behavior.csv to infected-c0-behavior.csv


Saving infected-c1-behavior.csv to infected-c1-behavior.csv


Saving infected-c2-behavior.csv to infected-c2-behavior.csv


Saving infected-c3-behavior.csv to infected-c3-behavior.csv


Saving infected-c4-behavior.csv to infected-c4-behavior.csv


Saving infected-c5-behavior.csv to infected-c5-behavior.csv


In [2]:
# ------------------------------
# DEFINE AUXILIARY METHODS
# ------------------------------

def map_results_to_category(results):
    # Goal: detect infected behavior; positive = infected, negative = normal
    true_positives = false_positives = true_negatives = false_negatives = 0
    for run in range(len(results)):
        data = results[run]
        elements = data[0]
        numbers = data[1]
        normal = data[2]
        # print("MAP", run, elements, numbers, infected)
        if len(elements) > 1:  # detected normal and infected samples
            if normal:
                true_negatives += numbers[0]
                false_positives += numbers[1]
            else:
                false_negatives += numbers[0]
                true_positives += numbers[1]
        else:  # detected only one type of samples
            if elements[0] == 0:  # detected normal
                if normal:
                    true_negatives += numbers[0]  # detected normal and is normal
                else:
                    false_negatives += numbers[0]  # detected normal but is infected
            else:  # detected infected
                if normal:
                    false_positives += numbers[0]  # detected infected but is normal
                else:
                    true_positives += numbers[0]  # detected infected and is infected
    return true_positives, true_negatives, false_positives, false_negatives


def calculate_f1_score(true_positives, true_negatives, false_positives, false_negatives):
    # print(true_positives, true_negatives, false_positives, false_negatives,
    #       true_positives + true_negatives + false_positives + false_negatives)
    precision = true_positives / (true_positives + false_positives)
    recall = true_positives / (true_positives + false_negatives)
    # print(precision, recall)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


def build_ml_model(model, contamination_factor=0.05):
  if model == "IF":
    from pyod.models.iforest import IForest
    clf = IForest(random_state=42, contamination=contamination_factor)
  elif model == "LOF":
    from pyod.models.lof import LOF
    clf = LOF(n_neighbors=50, contamination=contamination_factor)
  elif model == "OCSVM":
    from pyod.models.ocsvm import OCSVM
    clf = OCSVM(kernel='rbf', gamma=0.0001, nu=0.3, contamination=contamination_factor)
  else:
    clf = None  # provokes error on clf.train()
  return clf


In [4]:
# ------------------------------
# COMPUTE DETECTION RATES FOR DIFFERENT THRESHOLDS UNDER ISOLATION FOREST WITH ADVANCED PREPROCESSING
# ------------------------------

import pandas as pd
import numpy as np
import io

for threshold in [1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.91, 0.90]:
  # ==============================
  # PREPARE DATA
  # ==============================

  # READ DATA
  df_normal = pd.read_csv(io.BytesIO(normal_upl['normal-behavior.csv']))
  df_eval = pd.read_csv(io.BytesIO(eval_upl['eval-set.csv']))
  df_inf_c0 = pd.read_csv(io.BytesIO(inf_c0_upl['infected-c0-behavior.csv']))
  df_inf_c1 = pd.read_csv(io.BytesIO(inf_c1_upl['infected-c1-behavior.csv']))
  df_inf_c2 = pd.read_csv(io.BytesIO(inf_c2_upl['infected-c2-behavior.csv']))
  df_inf_c3 = pd.read_csv(io.BytesIO(inf_c3_upl['infected-c3-behavior.csv']))
  df_inf_c4 = pd.read_csv(io.BytesIO(inf_c4_upl['infected-c4-behavior.csv']))
  df_inf_c5 = pd.read_csv(io.BytesIO(inf_c5_upl['infected-c5-behavior.csv']))
  
  # DROP DUPLICATES
  df_normal.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_eval.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c0.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c1.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c2.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c3.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c4.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  df_inf_c5.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
  # DROP TIME FEATURES
  df_normal.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_eval.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c0.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c1.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c2.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c3.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c4.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  df_inf_c5.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
  # DROP SAMPLES WHERE RASPBERRY DID NOT HAVE CONNECTION
  df_normal = df_normal.loc[(df_normal['connectivity']==1)]
  df_eval = df_eval.loc[(df_eval['connectivity']==1)]
  df_inf_c0 = df_inf_c0.loc[(df_inf_c0['connectivity']==1)]
  df_inf_c1 = df_inf_c1.loc[(df_inf_c1['connectivity']==1)]
  df_inf_c2 = df_inf_c2.loc[(df_inf_c2['connectivity']==1)]
  df_inf_c3 = df_inf_c3.loc[(df_inf_c3['connectivity']==1)]
  df_inf_c4 = df_inf_c4.loc[(df_inf_c4['connectivity']==1)]
  df_inf_c5 = df_inf_c5.loc[(df_inf_c5['connectivity']==1)]

  # DROP CONSTANT FEATURES
  corr_const = df_normal.corr()
  all_labels = set(corr_const.keys())
  corr_const.dropna(axis=1, how="all", inplace=True)
  corr_const.reset_index(drop=True)
  cropped_labels = set(corr_const.keys())
  constant_feats = all_labels - cropped_labels

  df_normal.drop(constant_feats, inplace = True, axis = 1)
  df_eval.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c0.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c1.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c2.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c3.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c4.drop(constant_feats, inplace = True, axis = 1)
  df_inf_c5.drop(constant_feats, inplace = True, axis = 1)

  # DROP UNSTABLE FEATURES
  really_unsure = ["cpu_si", "tasks_running", "net:netif_rx", "sock:inet_sock_set_state", "tcp:tcp_destroy_sock", "writeback:global_dirty_state"]
  possibly_unstable_feats = ["net_eth_rx", "net_eth_tx", "cpu_temp", "ipi:ipi_raise", "kmem:mm_page_alloc_zone_locked", "kmem:mm_page_pcpu_drain", "rpm:rpm_suspend", "skb:consume_skb", "timer:timer_start", "workqueue:workqueue_activate_work", "writeback:writeback_pages_written"]
  unstable_feats = ["cpu_wa", "mem_free", "mem_used", "mem_cache", "swap_avail","net_lo_rx", "net_lo_tx", "filemap:mm_filemap_add_to_page_cache", "jbd2:jbd2_start_commit", "kmem:kfree", "kmem:kmalloc", "net:net_dev_queue", "net:net_dev_xmit", "qdisc:qdisc_dequeue", "random:mix_pool_bytes_nolock", "tcp:tcp_probe", "writeback:writeback_written"]

  df_normal.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_eval.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c0.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c1.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c2.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c3.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c4.drop(possibly_unstable_feats, inplace = True, axis = 1)
  df_inf_c5.drop(possibly_unstable_feats, inplace = True, axis = 1)

  df_normal.drop(unstable_feats, inplace = True, axis = 1)
  df_eval.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c0.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c1.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c2.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c3.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c4.drop(unstable_feats, inplace = True, axis = 1)
  df_inf_c5.drop(unstable_feats, inplace = True, axis = 1)
  
  # DROP HIGHLY CORRELATED FEATURES
  # https://www.projectpro.io/recipes/drop-out-highly-correlated-features-in-python
  # keeps first feature of correlated subset, so information is not lost
  corr_matrix = df_normal.corr().abs()
  upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
  correlated_feats = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
  # print("Remaining", len(corr_matrix)-len(correlated_feats), ", Correlated", len(correlated_feats), sorted(correlated_feats))
  df_normal.drop(correlated_feats, inplace = True, axis = 1)
  df_eval.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c0.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c1.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c2.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c3.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c4.drop(correlated_feats, inplace = True, axis = 1)
  df_inf_c5.drop(correlated_feats, inplace = True, axis = 1)

  # Reset index to plot
  df_normal.reset_index(inplace=True,drop=True)
  df_eval.reset_index(inplace=True,drop=True)
  df_inf_c0.reset_index(inplace=True,drop=True)
  df_inf_c1.reset_index(inplace=True,drop=True)
  df_inf_c2.reset_index(inplace=True,drop=True)
  df_inf_c3.reset_index(inplace=True,drop=True)
  df_inf_c4.reset_index(inplace=True,drop=True)
  df_inf_c5.reset_index(inplace=True,drop=True)

  # ==============================
  # APPLY MACHINE LEARNING
  # ==============================

  from sklearn.model_selection import train_test_split
  import numpy as np
  from scipy import stats
  df_normal = df_normal.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
  X_train = df_normal
  X_test = df_eval  # evaluate against test set
  # X_test = df_normal  # evaluate against training set

  # REMOVE OUTLIERS
  X_train=X_train[(np.abs(stats.zscore(X_train)) < 3).all(axis=1)]

  # SCALE DATASETS
  from sklearn.preprocessing import StandardScaler
  # Feature scaling
  scaler= StandardScaler().fit(X_train)
  # Transform training set
  X_train = scaler.transform(X_train)
  # Transform test set
  X_test = scaler.transform(X_test)
  # Transform inf_c0 set
  X_test_inf_c0 = scaler.transform(df_inf_c0)
  # Transform inf_c1 set
  X_test_inf_c1 = scaler.transform(df_inf_c1)
  # Transform inf_c2 set
  X_test_inf_c2 = scaler.transform(df_inf_c2)
  # Transform inf_c3 set
  X_test_inf_c3 = scaler.transform(df_inf_c3)
  # Transform inf_c4 set
  X_test_inf_c4 = scaler.transform(df_inf_c4)
  # Transform inf_c5 set
  X_test_inf_c5 = scaler.transform(df_inf_c5)

  contamination_factor=0.05

  # ----------
  # ISOLATION FOREST
  # ----------
  from pyod.models.iforest import IForest
    
  # TRAIN ML MODEL
  clf = IForest(random_state=42, contamination=contamination_factor)
  # Model training
  clf.fit(X_train)

  # EVALUATE DATASETS
  all_results = []

  # Model evaluation with good behaviour
  pred=clf.predict(X_test)
  unique_elements_normal, counts_elements_normal = np.unique(pred, return_counts=True)
  perc_normal = "%.2f" % (counts_elements_normal[0] / (counts_elements_normal[0] + counts_elements_normal[1]) * 100)
  all_results.append([unique_elements_normal, counts_elements_normal, True])

  # Model evaluation with bad behavior
  pred=clf.predict(X_test_inf_c0)
  unique_elements_0, counts_elements_0 = np.unique(pred, return_counts=True)
  perc_0 = "%.2f" % (counts_elements_0[0] / (counts_elements_0[0] + counts_elements_0[1]) * 100)
  all_results.append([unique_elements_0, counts_elements_0, False])

  pred=clf.predict(X_test_inf_c1)
  unique_elements_1, counts_elements_1 = np.unique(pred, return_counts=True)
  if len(counts_elements_1) > 1:
    perc_1 = (counts_elements_1[0] / (counts_elements_1[0] + counts_elements_1[1]) * 100)
  elif unique_elements_1[0] == 1:  # detected
    perc_1 = 0
  else:  # hidden
    perc_1 = 100
  all_results.append([unique_elements_1, counts_elements_1, False])

  pred=clf.predict(X_test_inf_c2)
  unique_elements_2, counts_elements_2 = np.unique(pred, return_counts=True)
  if len(counts_elements_2) > 1:
    perc_2 = (counts_elements_2[0] / (counts_elements_2[0] + counts_elements_2[1]) * 100)
  elif unique_elements_2[0] == 1:  # detected
    perc_2 = 0
  else:  # hidden
    perc_2 = 100
  all_results.append([unique_elements_2, counts_elements_2, False])

  pred=clf.predict(X_test_inf_c3)
  unique_elements_3, counts_elements_3 = np.unique(pred, return_counts=True)
  if len(counts_elements_3) > 1:
    perc_3 = (counts_elements_3[0] / (counts_elements_3[0] + counts_elements_3[1]) * 100)
  elif unique_elements_3[0] == 1:  # detected
    perc_3 = 0
  else:  # hidden
    perc_3 = 100
  all_results.append([unique_elements_3, counts_elements_3, False])

  pred=clf.predict(X_test_inf_c4)
  unique_elements_4, counts_elements_4 = np.unique(pred, return_counts=True)
  if len(counts_elements_4) > 1:
    perc_4 = (counts_elements_4[0] / (counts_elements_4[0] + counts_elements_4[1]) * 100)
  elif unique_elements_4[0] == 1:  # detected
    perc_4 = 0
  else:  # hidden
    perc_4 = 100
  all_results.append([unique_elements_4, counts_elements_4, False])

  pred=clf.predict(X_test_inf_c5)
  unique_elements_5, counts_elements_5 = np.unique(pred, return_counts=True)
  if len(counts_elements_5) > 1:
    perc_5 = (counts_elements_5[0] / (counts_elements_5[0] + counts_elements_5[1]) * 100)
  elif unique_elements_5[0] == 1:  # detected
    perc_5 = 0
  else:  # hidden
    perc_5 = 100
  all_results.append([unique_elements_5, counts_elements_5, False])

  # CALCULATE F1 SCORE
  f1 = calculate_f1_score(*map_results_to_category(all_results))

  # PRINT RESULTS FOR THIS THRESHOLD
  print("%.2f" % threshold, perc_normal, perc_0, "%.2f" % perc_1, "%.2f" % perc_2, "%.2f" % perc_3, "%.2f" % perc_4, "%.2f" % perc_5, "F1: %.5f" % f1, "Feats: {}".format(len(corr_matrix)-len(correlated_feats)), sep="\t")

1.00	88.39	89.75	0.00	0.00	80.36	80.11	76.75	F1: 0.52884	Feats: 58
0.99	88.94	91.62	0.62	0.21	80.18	82.05	77.38	F1: 0.51939	Feats: 50
0.98	85.83	88.50	0.00	0.00	73.57	75.91	65.62	F1: 0.56782	Feats: 43
0.97	86.44	88.25	0.00	0.00	68.57	69.89	62.38	F1: 0.59470	Feats: 42
0.96	83.72	88.00	0.00	0.00	56.96	59.32	56.62	F1: 0.63606	Feats: 40
0.95	83.67	85.62	0.00	0.00	54.29	58.41	52.88	F1: 0.65138	Feats: 39
0.94	85.72	86.38	0.00	0.00	63.21	69.43	57.12	F1: 0.61426	Feats: 38
0.93	84.83	86.25	0.00	0.00	61.96	65.91	59.50	F1: 0.61715	Feats: 37
0.92	82.89	84.38	0.00	0.00	60.00	66.48	56.75	F1: 0.62320	Feats: 36
0.91	82.56	85.38	0.00	0.00	46.61	55.91	52.00	F1: 0.66481	Feats: 35
0.90	84.11	88.12	0.00	0.00	68.04	70.91	60.75	F1: 0.59245	Feats: 34


In [5]:
# ------------------------------
# COMPUTE DETECTION RATES FOR DIFFERENT THRESHOLDS UNDER DIFFERENT ML MODELS
# ------------------------------

import pandas as pd
import numpy as np
import io

for model in ["IF", "OCSVM", "LOF"]:
  print("\n" + model)

  for advanced_preprocessor in [False, True]:
    print("Advanced" if advanced_preprocessor else "Simple")

    for threshold in [1.00, 0.99, 0.98, 0.97, 0.96, 0.95, 0.94, 0.93, 0.92, 0.91, 0.90]:
      # ==============================
      # PREPARE DATA
      # ==============================

      # READ DATA
      df_normal = pd.read_csv(io.BytesIO(normal_upl['normal-behavior.csv']))
      df_eval = pd.read_csv(io.BytesIO(eval_upl['eval-set.csv']))
      df_inf_c0 = pd.read_csv(io.BytesIO(inf_c0_upl['infected-c0-behavior.csv']))
      df_inf_c1 = pd.read_csv(io.BytesIO(inf_c1_upl['infected-c1-behavior.csv']))
      df_inf_c2 = pd.read_csv(io.BytesIO(inf_c2_upl['infected-c2-behavior.csv']))
      df_inf_c3 = pd.read_csv(io.BytesIO(inf_c3_upl['infected-c3-behavior.csv']))
      df_inf_c4 = pd.read_csv(io.BytesIO(inf_c4_upl['infected-c4-behavior.csv']))
      df_inf_c5 = pd.read_csv(io.BytesIO(inf_c5_upl['infected-c5-behavior.csv']))
      
      # DROP DUPLICATES
      df_normal.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_eval.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c0.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c1.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c2.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c3.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c4.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      df_inf_c5.drop(['qdisc:qdisc_dequeue.1','skb:consume_skb.1', 'skb:kfree_skb.1'], inplace = True, axis = 1)
      # DROP TIME FEATURES
      df_normal.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_eval.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c0.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c1.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c2.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c3.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c4.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      df_inf_c5.drop(['time','timestamp', 'seconds'], inplace = True, axis = 1)
      # DROP SAMPLES WHERE RASPBERRY DID NOT HAVE CONNECTION
      df_normal = df_normal.loc[(df_normal['connectivity']==1)]
      df_eval = df_eval.loc[(df_eval['connectivity']==1)]
      df_inf_c0 = df_inf_c0.loc[(df_inf_c0['connectivity']==1)]
      df_inf_c1 = df_inf_c1.loc[(df_inf_c1['connectivity']==1)]
      df_inf_c2 = df_inf_c2.loc[(df_inf_c2['connectivity']==1)]
      df_inf_c3 = df_inf_c3.loc[(df_inf_c3['connectivity']==1)]
      df_inf_c4 = df_inf_c4.loc[(df_inf_c4['connectivity']==1)]
      df_inf_c5 = df_inf_c5.loc[(df_inf_c5['connectivity']==1)]

      # DROP CONSTANT FEATURES
      corr_const = df_normal.corr()
      all_labels = set(corr_const.keys())
      corr_const.dropna(axis=1, how="all", inplace=True)
      corr_const.reset_index(drop=True)
      cropped_labels = set(corr_const.keys())
      constant_feats = all_labels - cropped_labels

      df_normal.drop(constant_feats, inplace = True, axis = 1)
      df_eval.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c0.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c1.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c2.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c3.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c4.drop(constant_feats, inplace = True, axis = 1)
      df_inf_c5.drop(constant_feats, inplace = True, axis = 1)

      if advanced_preprocessor:
        # DROP UNSTABLE FEATURES
        really_unsure = ["cpu_si", "tasks_running", "net:netif_rx", "sock:inet_sock_set_state", "tcp:tcp_destroy_sock", "writeback:global_dirty_state"]
        possibly_unstable_feats = ["net_eth_rx", "net_eth_tx", "cpu_temp", "ipi:ipi_raise", "kmem:mm_page_alloc_zone_locked", "kmem:mm_page_pcpu_drain", "rpm:rpm_suspend", "skb:consume_skb", "timer:timer_start", "workqueue:workqueue_activate_work", "writeback:writeback_pages_written"]
        unstable_feats = ["cpu_wa", "mem_free", "mem_used", "mem_cache", "swap_avail","net_lo_rx", "net_lo_tx", "filemap:mm_filemap_add_to_page_cache", "jbd2:jbd2_start_commit", "kmem:kfree", "kmem:kmalloc", "net:net_dev_queue", "net:net_dev_xmit", "qdisc:qdisc_dequeue", "random:mix_pool_bytes_nolock", "tcp:tcp_probe", "writeback:writeback_written"]

        df_normal.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_eval.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c0.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c1.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c2.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c3.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c4.drop(possibly_unstable_feats, inplace = True, axis = 1)
        df_inf_c5.drop(possibly_unstable_feats, inplace = True, axis = 1)

        df_normal.drop(unstable_feats, inplace = True, axis = 1)
        df_eval.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c0.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c1.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c2.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c3.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c4.drop(unstable_feats, inplace = True, axis = 1)
        df_inf_c5.drop(unstable_feats, inplace = True, axis = 1)
        
        # DROP HIGHLY CORRELATED FEATURES
        # https://www.projectpro.io/recipes/drop-out-highly-correlated-features-in-python
        corr_matrix = df_normal.corr().abs()
        upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
        correlated_feats = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
        # print("Remaining", len(corr_matrix)-len(correlated_feats), ", Correlated", len(correlated_feats), sorted(correlated_feats))
        df_normal.drop(correlated_feats, inplace = True, axis = 1)
        df_eval.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c0.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c1.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c2.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c3.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c4.drop(correlated_feats, inplace = True, axis = 1)
        df_inf_c5.drop(correlated_feats, inplace = True, axis = 1)
      else:
        # compute to print number of features
        corr_matrix = df_normal.corr().abs()
        correlated_feats = []

      # Reset index to plot
      df_normal.reset_index(inplace=True,drop=True)
      df_eval.reset_index(inplace=True,drop=True)
      df_inf_c0.reset_index(inplace=True,drop=True)
      df_inf_c1.reset_index(inplace=True,drop=True)
      df_inf_c2.reset_index(inplace=True,drop=True)
      df_inf_c3.reset_index(inplace=True,drop=True)
      df_inf_c4.reset_index(inplace=True,drop=True)
      df_inf_c5.reset_index(inplace=True,drop=True)

      # ==============================
      # APPLY MACHINE LEARNING
      # ==============================

      from sklearn.model_selection import train_test_split
      import numpy as np
      from scipy import stats
      df_normal = df_normal.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle
      X_train = df_normal
      X_test = df_eval  # evaluate against test set
      # X_test = df_normal  # evaluate against training set

      # REMOVE OUTLIERS
      X_train=X_train[(np.abs(stats.zscore(X_train)) < 3).all(axis=1)]

      # SCALE DATASETS
      from sklearn.preprocessing import StandardScaler
      # Feature scaling
      scaler= StandardScaler().fit(X_train)
      # Transform training set
      X_train = scaler.transform(X_train)
      # Transform test set
      X_test = scaler.transform(X_test)
      # Transform inf_c0 set
      X_test_inf_c0 = scaler.transform(df_inf_c0)
      # Transform inf_c1 set
      X_test_inf_c1 = scaler.transform(df_inf_c1)
      # Transform inf_c2 set
      X_test_inf_c2 = scaler.transform(df_inf_c2)
      # Transform inf_c3 set
      X_test_inf_c3 = scaler.transform(df_inf_c3)
      # Transform inf_c4 set
      X_test_inf_c4 = scaler.transform(df_inf_c4)
      # Transform inf_c5 set
      X_test_inf_c5 = scaler.transform(df_inf_c5)

      # ----------
      # APPLY ML MODEL
      # ----------

      contamination_factor=0.05

      # TRAIN ML MODEL
      clf = build_ml_model(model, contamination_factor)
      # Model training
      clf.fit(X_train)

      # EVALUATE DATASETS
      all_results = []

      # Model evaluation with good behaviour
      pred=clf.predict(X_test)
      unique_elements_normal, counts_elements_normal = np.unique(pred, return_counts=True)
      perc_normal = "%.2f" % (counts_elements_normal[0] / (counts_elements_normal[0] + counts_elements_normal[1]) * 100)
      all_results.append([unique_elements_normal, counts_elements_normal, True])

      # Model evaluation with bad behavior
      pred=clf.predict(X_test_inf_c0)
      unique_elements_0, counts_elements_0 = np.unique(pred, return_counts=True)
      perc_0 = "%.2f" % (counts_elements_0[0] / (counts_elements_0[0] + counts_elements_0[1]) * 100)
      all_results.append([unique_elements_0, counts_elements_0, False])

      pred=clf.predict(X_test_inf_c1)
      unique_elements_1, counts_elements_1 = np.unique(pred, return_counts=True)
      if len(counts_elements_1) > 1:
        perc_1 = (counts_elements_1[0] / (counts_elements_1[0] + counts_elements_1[1]) * 100)
      elif unique_elements_1[0] == 1:  # detected
        perc_1 = 0
      else:  # hidden
        perc_1 = 100
      all_results.append([unique_elements_1, counts_elements_1, False])

      pred=clf.predict(X_test_inf_c2)
      unique_elements_2, counts_elements_2 = np.unique(pred, return_counts=True)
      if len(counts_elements_2) > 1:
        perc_2 = (counts_elements_2[0] / (counts_elements_2[0] + counts_elements_2[1]) * 100)
      elif unique_elements_2[0] == 1:  # detected
        perc_2 = 0
      else:  # hidden
        perc_2 = 100
      all_results.append([unique_elements_2, counts_elements_2, False])

      pred=clf.predict(X_test_inf_c3)
      unique_elements_3, counts_elements_3 = np.unique(pred, return_counts=True)
      if len(counts_elements_3) > 1:
        perc_3 = (counts_elements_3[0] / (counts_elements_3[0] + counts_elements_3[1]) * 100)
      elif unique_elements_3[0] == 1:  # detected
        perc_3 = 0
      else:  # hidden
        perc_3 = 100
      all_results.append([unique_elements_3, counts_elements_3, False])

      pred=clf.predict(X_test_inf_c4)
      unique_elements_4, counts_elements_4 = np.unique(pred, return_counts=True)
      if len(counts_elements_4) > 1:
        perc_4 = (counts_elements_4[0] / (counts_elements_4[0] + counts_elements_4[1]) * 100)
      elif unique_elements_4[0] == 1:  # detected
        perc_4 = 0
      else:  # hidden
        perc_4 = 100
      all_results.append([unique_elements_4, counts_elements_4, False])

      pred=clf.predict(X_test_inf_c5)
      unique_elements_5, counts_elements_5 = np.unique(pred, return_counts=True)
      if len(counts_elements_5) > 1:
        perc_5 = (counts_elements_5[0] / (counts_elements_5[0] + counts_elements_5[1]) * 100)
      elif unique_elements_5[0] == 1:  # detected
        perc_5 = 0
      else:  # hidden
        perc_5 = 100
      all_results.append([unique_elements_5, counts_elements_5, False])

      # CALCULATE F1 SCORE
      f1 = calculate_f1_score(*map_results_to_category(all_results))

      # PRINT RESULTS FOR THIS THRESHOLD
      print("%.2f" % threshold, perc_normal, perc_0, "%.2f" % perc_1, "%.2f" % perc_2, "%.2f" % perc_3, "%.2f" % perc_4, "%.2f" % perc_5, "F1: %.5f" % f1, "Feats: {}".format(len(corr_matrix)-len(correlated_feats)), sep="\t")
      
      if not advanced_preprocessor:
        break  # only need to print once, no threshold involved



IF
Simple
1.00	85.22	79.50	0.00	0.00	65.71	66.93	59.50	F1: 0.62320	Feats: 86
Advanced
1.00	88.39	89.75	0.00	0.00	80.36	80.11	76.75	F1: 0.52884	Feats: 58
0.99	88.94	91.62	0.62	0.21	80.18	82.05	77.38	F1: 0.51939	Feats: 50
0.98	85.83	88.50	0.00	0.00	73.57	75.91	65.62	F1: 0.56782	Feats: 43
0.97	86.44	88.25	0.00	0.00	68.57	69.89	62.38	F1: 0.59470	Feats: 42
0.96	83.72	88.00	0.00	0.00	56.96	59.32	56.62	F1: 0.63606	Feats: 40
0.95	83.67	85.62	0.00	0.00	54.29	58.41	52.88	F1: 0.65138	Feats: 39
0.94	85.72	86.38	0.00	0.00	63.21	69.43	57.12	F1: 0.61426	Feats: 38
0.93	84.83	86.25	0.00	0.00	61.96	65.91	59.50	F1: 0.61715	Feats: 37
0.92	82.89	84.38	0.00	0.00	60.00	66.48	56.75	F1: 0.62320	Feats: 36
0.91	82.56	85.38	0.00	0.00	46.61	55.91	52.00	F1: 0.66481	Feats: 35
0.90	84.11	88.12	0.00	0.00	68.04	70.91	60.75	F1: 0.59245	Feats: 34

OCSVM
Simple
1.00	69.94	68.50	0.00	0.00	5.36	21.70	21.25	F1: 0.80531	Feats: 86
Advanced
1.00	74.50	70.38	0.00	0.00	6.43	22.50	23.12	F1: 0.80728	Feats: 58
0.99	74.83	71.25	0.00